# U-Net 폐 분할 모델 학습
- **목적:** `experiment.ipynb`의 `seg_crop` 전처리 변형(폐 bbox crop)에 쓸 폐 분할 모델을 만든다.
- **데이터:** 이 데이터셋(Chest X-Ray Pneumonia)에는 폐 마스크가 없으므로, 공개 폐 마스크 데이터 [Chest X-ray Masks and Labels](https://www.kaggle.com/datasets/nikhilpandey360/chest-xray-masks-and-labels)(Montgomery 138장 + Shenzhen 566장, 성인)를 사용한다. 폐렴 데이터의 라벨 / val / test는 사용하지 않는다(누수 없음).
    - `./data_seg`에 받아두면 그 경로를, 없으면 kagglehub로 다운로드한다(압축 약 10GB: 같은 데이터가 두 벌 들어 있음).
- **모델:** `segmentation_models_pytorch.Unet`(encoder: ImageNet 사전학습 resnet34, 1채널 입력은 첫 conv의 RGB 가중치 합산), BCE + Dice loss.
- **산출물:** `results/seg/unet_lung.pt` (val Dice 최고 에폭). 품질 확인은 `eda.ipynb` 3-6(b), 전체 bbox 추론은 `experiment.ipynb` 1장에서 한다.
- colab: `%pip install -q segmentation-models-pytorch`

In [ ]:
# ---------- 설정 ----------
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
SEG_SIZE = 256                                   # U-Net 입력 크기 (비율 유지 letterbox)
SEG_EPOCHS, SEG_BATCH, SEG_LR = 25, 16, 1e-3
VAL_RATIO = 0.1
ROT_DEG, HFLIP_P = 10, 0.5                        # 증강 (experiment.ipynb와 같은 값)
GRAY_MEAN = float(np.mean([0.485, 0.456, 0.406]))   # 1채널 정규화: ImageNet 3채널 통계의 평균
GRAY_STD = float(np.mean([0.229, 0.224, 0.225]))

SEG_KAGGLE = "nikhilpandey360/chest-xray-masks-and-labels"
SEG_DATA_DIR = Path("./data_seg")                 # 직접 받아둔 경우 이 위치, 없으면 kagglehub 캐시로 다운로드
SEG_DIR = Path("./results/seg")
SEG_DIR.mkdir(parents=True, exist_ok=True)
SEG_WEIGHTS = SEG_DIR / "unet_lung.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"device={DEVICE}")


def letterbox(img: np.ndarray, size: int, interp=cv2.INTER_AREA) -> np.ndarray:
    """긴 변을 size에 맞추고 0으로 패딩한다."""
    h, w = img.shape[:2]
    s = size / max(w, h)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    ox, oy = (size - nw) // 2, (size - nh) // 2
    out = np.zeros((size, size), dtype=img.dtype)
    out[oy:oy + nh, ox:ox + nw] = cv2.resize(img, (nw, nh), interpolation=interp)
    return out


def seg_normalize(x_uint8: torch.Tensor) -> torch.Tensor:
    return (x_uint8.float() / 255 - GRAY_MEAN) / GRAY_STD


def build_seg_unet(pretrained: bool = True) -> nn.Module:
    # in_channels=1: smp가 ImageNet 첫 conv의 RGB 가중치를 합산해 1채널로 만든다
    return smp.Unet("resnet34", encoder_weights="imagenet" if pretrained else None, in_channels=1, classes=1)

In [ ]:
# ---------- 데이터: 이미지-마스크 짝짓기 -> 256 letterbox -> 9:1 분할 ----------
def load_seg_pairs() -> pd.DataFrame:
    """CXR_png 이미지와 masks 마스크를 파일명으로 짝짓는다. Shenzhen 마스크는 '_mask' 접미사가 붙어 있고, 일부 이미지는 마스크가 없다."""
    root = SEG_DATA_DIR if SEG_DATA_DIR.exists() else Path(__import__("kagglehub").dataset_download(SEG_KAGGLE))
    img_dir = next(root.rglob("CXR_png"))
    mask_dir = next(root.rglob("masks"))
    masks = {p.stem.removesuffix("_mask"): p for p in mask_dir.glob("*.png")}
    rows = [{"image": p, "mask": masks[p.stem], "source": "shenzhen" if p.stem.startswith("CHNCXR") else "montgomery"}
            for p in sorted(img_dir.glob("*.png")) if p.stem in masks]
    return pd.DataFrame(rows)


def _load_seg_one(row):
    gray = cv2.imread(str(row.image), cv2.IMREAD_GRAYSCALE)
    mask = (cv2.imread(str(row.mask), cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    return letterbox(gray, SEG_SIZE), letterbox(mask, SEG_SIZE, cv2.INTER_NEAREST)


pairs = load_seg_pairs()
print("짝지은 이미지:", pairs["source"].value_counts().to_dict())
t0 = time.time()
with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:   # 원본이 3000px 안팎이라 디코딩을 병렬화한다
    out = list(ex.map(_load_seg_one, pairs.itertuples()))
X = torch.from_numpy(np.stack([o[0] for o in out])).unsqueeze(1).to(DEVICE)           # uint8 (N,1,256,256)
M = torch.from_numpy(np.stack([o[1] for o in out])).unsqueeze(1).float().to(DEVICE)   # 0/1 (N,1,256,256)

perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(SEED))
n_val = round(len(X) * VAL_RATIO)
val_idx, tr_idx = perm[:n_val], perm[n_val:]
print(f"train {len(tr_idx)} / val {len(val_idx)}, 로드 {time.time() - t0:.0f}s")

In [ ]:
# ---------- 학습: val Dice 최고 에폭을 저장 (가중치가 이미 있으면 건너뜀, 다시 학습하려면 FORCE_TRAIN = True) ----------
FORCE_TRAIN = False


def seg_augment(x, m, gen):
    """이미지/마스크에 같은 샘플별 affine(±ROT_DEG 회전 + 좌우반전)을 적용한다. 마스크는 nearest."""
    b = x.size(0)
    angle = (torch.rand(b, generator=gen) * 2 - 1) * np.deg2rad(ROT_DEG)
    flip = torch.where(torch.rand(b, generator=gen) < HFLIP_P, -1.0, 1.0)
    cos, sin = torch.cos(angle), torch.sin(angle)
    theta = torch.zeros(b, 2, 3)
    theta[:, 0, 0], theta[:, 0, 1] = cos * flip, -sin
    theta[:, 1, 0], theta[:, 1, 1] = sin * flip, cos
    grid = F.affine_grid(theta.to(x.device), list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    m = F.grid_sample(m, grid, mode="nearest", padding_mode="zeros", align_corners=False)
    return x, m


@torch.no_grad()
def dice_iou(logits, target):
    pred = (logits > 0).float()
    inter = (pred * target).sum((1, 2, 3))
    total = pred.sum((1, 2, 3)) + target.sum((1, 2, 3))
    dice = 2 * inter / total.clamp(min=1)
    iou = inter / (total - inter).clamp(min=1)
    return dice, iou   # 이미지별


@torch.no_grad()
def predict_logits(model, idx):
    model.eval()
    return torch.cat([model(seg_normalize(X[idx[s:s + SEG_BATCH].to(DEVICE)])) for s in range(0, len(idx), SEG_BATCH)])


def train_seg_unet():
    torch.manual_seed(SEED)
    model = build_seg_unet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=SEG_LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SEG_EPOCHS)
    dice_loss = smp.losses.DiceLoss("binary", from_logits=True)
    bce = nn.BCEWithLogitsLoss()
    gen = torch.Generator().manual_seed(SEED)
    best_dice = -1.0
    for epoch in range(1, SEG_EPOCHS + 1):
        t0 = time.time()
        model.train()
        order = tr_idx[torch.randperm(len(tr_idx), generator=gen)]
        total = 0.0
        for s in range(0, len(order), SEG_BATCH):
            idx = order[s:s + SEG_BATCH].to(DEVICE)
            x, m = seg_augment(seg_normalize(X[idx]), M[idx], gen)
            logits = model(x)
            loss = bce(logits, m) + dice_loss(logits, m)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
        scheduler.step()

        dice, iou = dice_iou(predict_logits(model, val_idx), M[val_idx.to(DEVICE)])
        dice, iou = dice.mean().item(), iou.mean().item()
        improved = dice > best_dice
        if improved:
            best_dice = dice
            torch.save({"state_dict": model.state_dict(), "epoch": epoch, "val_dice": dice, "val_iou": iou}, SEG_WEIGHTS)
        print(f"[unet] ep {epoch:02d} | train loss {total / len(tr_idx):.4f} | val dice {dice:.4f} iou {iou:.4f} | "
              f"{time.time() - t0:.0f}s" + (" *" if improved else ""))


if FORCE_TRAIN or not SEG_WEIGHTS.exists():
    train_seg_unet()
ckpt = torch.load(SEG_WEIGHTS, map_location="cpu")
seg_model = build_seg_unet(pretrained=False).to(DEVICE).eval()
seg_model.load_state_dict(ckpt["state_dict"])
print(f"best: epoch {ckpt['epoch']}, val dice {ckpt['val_dice']:.4f}, iou {ckpt['val_iou']:.4f} -> {SEG_WEIGHTS}")

In [ ]:
# ---------- val 확인: 이미지별 Dice 분포 + Dice 낮은 순 / 무작위 예측 (빨강: 정답, 노랑: 예측) ----------
logits = predict_logits(seg_model, val_idx)
dice, _ = dice_iou(logits, M[val_idx.to(DEVICE)])
dice = dice.cpu().numpy()
print(f"val 이미지별 Dice: mean {dice.mean():.4f}, min {dice.min():.4f}, 하위 5% {np.percentile(dice, 5):.4f}")

shown = list(np.argsort(dice)[:6]) + list(np.random.default_rng(SEED).choice(len(dice), 6, replace=False))
fig, axes = plt.subplots(2, 6, figsize=(15, 5.6))
for ax, i in zip(axes.flat, shown):
    rgb = cv2.cvtColor(X[val_idx[i]].squeeze().cpu().numpy(), cv2.COLOR_GRAY2RGB)
    for mask, color in [(M[val_idx[i]].squeeze().cpu().numpy().astype(np.uint8), (255, 0, 0)),
                        ((logits[i, 0] > 0).cpu().numpy().astype(np.uint8), (255, 220, 0))]:
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(rgb, contours, -1, color, 1)
    ax.imshow(rgb)
    ax.set_title(f"{pairs['source'].iloc[int(val_idx[i])]}  dice {dice[i]:.3f}", fontsize=8)
    ax.axis("off")
axes[0, 0].text(-0.08, 0.5, "lowest dice", transform=axes[0, 0].transAxes, ha="right", va="center", rotation=90)
axes[1, 0].text(-0.08, 0.5, "random", transform=axes[1, 0].transAxes, ha="right", va="center", rotation=90)
plt.suptitle("U-Net val predictions (red: ground truth, yellow: prediction)")
plt.tight_layout()
plt.show()